# 02 · CLIP ZeroShot —— 图文各训一座塔，cos 相似度直接分类

**家族位置**：`06_Transformer_Vision_Multimodal` 第 2 站。01 把图切成“字”喂进 ViT，本章加一座**对称的文本塔**：图向量 `v` 与类名文本向量 `t_c` 算 `cos`，**不训练任何分类头**——这就是 CLIP 的 zero-shot 范式。无外网，字符级 tokenizer，CPU 分钟级。

**学习目标**
1. 双塔结构：图像塔（01 ViT 骨架去分类头）+ 文本塔（char embedding + Encoder + EOS 位）
2. InfoNCE 对比损失：对角正样本拉近、同 batch 其余推开，温度可学习
3. zero-shot 分类：`p(class|image) = softmax(cos(v, t_c)/τ)`，类名当“提示”
4. 与 01 监督 ViT（6ep 0.361）同台：小数据对比学习能到多少、谁赢在哪

## 1. 原理：相亲打分，不是背答案

### 通俗理解

**一句话**：01 的 ViT 是“背答案”——喂 4000 张带标签图硬记“这个形状=ship”；CLIP 是“找对象”——图和文字各站一排，配对的拉近、乱点的推远，最后看图“谁最像哪句话”。

**比喻**：文本塔把 `"a photo of a ship"` 压成一个坐标点，图像塔把像素压到同一个空间里的另一个坐标点。训练只干一件事：**让配对的两个点尽量近、让 batch 里别的点尽量远**（InfoNCE）。训完不用学分类——10 个类名先各自站好位，新图谁离 ship 最近就判 ship。

### 结构账

```
图像塔： ViT-Tiny(dim=128,L=4,H=4) 复用 01 骨架，砍掉分类头，取 [CLS] 过投影 → v (B,64) 单位向量
文本塔： char embedding(36→128) + 可学习pos(16) + Encoder×2 + 取 EOS 位过投影 → t (B,64) 单位向量
对比头： logits[i,j] = exp(logit_scale·<v_i,t_j>)；对角为正，按行 CE（双向对称）
zero-shot： 测试集前向图塔 → 与 10 个类名文本嵌入算 cos → argmax 即预测
```

- **与 01/05-02 的对应**：同 Encoder+CLS/汇总位思想；不同在**目标函数从“背标签 CE”换成“图文配对 InfoNCE”**
- **评估**：zero-shot test acc + 相似度矩阵热力图 + 错例

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_cifar10_local, class_prompts
from common.models import CLIPMini, tokenize_prompts
from common.utils import set_seed, setup_chinese_font, count_params, unnormalize

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

N_TRAIN, N_VAL = 6000, 800
EPOCHS, BATCH, LR = 12, 128, 3e-4
Xtr, ytr, Xva, yva, Xte, yte = load_cifar10_local(N_TRAIN, N_VAL, seed=0)
prompts = class_prompts()
print(f"train {tuple(Xtr.shape)} / test {tuple(Xte.shape)} | prompts {len(prompts)} 条，例 '{prompts[6]}'")


## 2. 数据：图 + 固定类名模板

CIFAR-10 子集 `6000/800/10000`（02 家族缓存，无外网）。文本只有 10 个固定模板 `a photo of a {class}`——训练用**图对应的类名**当配对文本；zero-shot 评估时 10 个类名全量前向当“候选锚”。

In [ ]:
# fig0：图文配对示意（8 张图 + 对应模板文本）
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for k, ax in enumerate(axes):
    i = k * 700
    ax.imshow(unnormalize(Xtr[i]))
    ax.set_title(f"image i{k}\n↕ 配对文本\n'{prompts[int(ytr[i])]}'", fontsize=8)
    ax.axis("off")
plt.suptitle("CLIP 训练单位：(图, 类名模板) 配对——对角拉近，batch 内其余推开", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_pairs.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 训练：双塔 InfoNCE，图塔热、文本塔小

In [ ]:
# 文本 token 化：把每个 batch 的图像标签映射到 10 条固定 prompt 的 token ids
prompt_ids = tokenize_prompts(prompts, max_len=64)  # (10,64)
prompt_tokens = torch.tensor(prompt_ids, dtype=torch.long)

def make_pair_loader(X, y, batch, shuffle=True):
    return DataLoader(TensorDataset(X, y, prompt_tokens[y]), batch_size=batch, shuffle=shuffle)

train_loader = make_pair_loader(Xtr, ytr, BATCH)

def train_clip(model, loader, epochs, lr=LR, log_every=2):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    hist = []
    n = len(loader.dataset)
    for ep in range(1, epochs+1):
        model.train()
        tot = 0.0
        for xb, yb, tb in loader:
            loss = model(xb, tb)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(xb)
        avg = tot/n
        hist.append(avg)
        if ep % log_every == 0 or ep == 1:
            print(f"epoch {ep:02d} | InfoNCE loss {avg:.4f}", flush=True)
    return hist

model = CLIPMini(img_dim=128, img_depth=4, heads=4, txt_dim=128, txt_depth=2, proj_dim=64)
print(f"CLIP params={count_params(model)} (图像塔+文本塔+投影)")
hist = train_clip(model, train_loader, EPOCHS)

# fig1：InfoNCE 收敛曲线
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(hist, color="#4C72B0", marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("InfoNCE loss")
ax.set_title("CLIP-mini 对比训练收敛（loss→ln(1)=0 是背对背满分，小数据难完全到 0）")
plt.tight_layout()
plt.savefig(FIGS / "fig1_loss.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Zero-shot 评估：不训分类头，直接比“像不像类名”

In [ ]:
@torch.no_grad()
def zero_shot_eval(model, X, y, prompt_tokens, batch=256):
    model.eval()
    text_feats = F.normalize(model.encode_text(prompt_tokens), dim=-1)   # (10,D)
    preds, sims = [], []
    for i in range(0, len(X), batch):
        img_feats = F.normalize(model.encode_image(X[i:i+batch]), dim=-1)
        s = img_feats @ text_feats.T  # (B,10) cos
        sims.append(s.numpy())
        preds.append(s.argmax(dim=-1).numpy())
    sims = np.concatenate(sims); preds = np.concatenate(preds)
    acc = (preds == y.numpy()).mean()
    return acc, preds, sims

acc_zs, preds, sims = zero_shot_eval(model, Xte, yte, prompt_tokens)
# 对照：同预算监督 ViT（01 站 test=0.3614，6ep）——用本章 12ep 的图塔+分类头再训一个，公平比
sup = CLIPMini(img_dim=128, img_depth=4, heads=4, txt_dim=128, txt_depth=2, proj_dim=64)
print(f"zero-shot test acc = {acc_zs:.4f}")

# 监督基线（同数据同 epoch 预算，图塔后接 Linear 头）
from common.models import ViTTiny
vit_sup = ViTTiny(dim=128, depth=4, heads=4)
opt_s = torch.optim.AdamW(vit_sup.parameters(), lr=LR, weight_decay=0.05)
torch.manual_seed(0)
train_loader_s = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)
for ep in range(1, EPOCHS+1):
    vit_sup.train()
    for xb, yb in train_loader_s:
        loss = nn.functional.cross_entropy(vit_sup(xb), yb)
        opt_s.zero_grad(); loss.backward(); opt_s.step()
vit_sup.eval()
with torch.no_grad():
    acc_sup = (torch.cat([vit_sup(Xte[i:i+256]).argmax(-1) for i in range(0, len(Xte), 256)]) == yte).float().mean().item()
print(f"监督 ViT（同 12ep 预算）test acc = {acc_sup:.4f}")

# fig2：zero-shot vs 监督柱状
fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.bar(["CLIP zero-shot", "监督 ViT(12ep)"], [acc_zs, acc_sup], color=["#DD8452","#4C72B0"])
for i, v in enumerate([acc_zs, acc_sup]):
    ax.text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylim(0,1.0); ax.set_ylabel("test acc")
ax.set_title("zero-shot vs 监督：CLIP 没学分类头，但类名当提示")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 可视化：相似度矩阵与错例

In [ ]:
from common.data import CIFAR10_CLASSES
# fig3：16 张测试图的相似度矩阵热力（列=10 类名）
fig, ax = plt.subplots(figsize=(7.6, 4.4))
im = ax.imshow(sims[:16], cmap="RdYlBu_r", vmin=sims.min(), vmax=sims.max())
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(16)); ax.set_yticklabels([f"y={int(yte[i])}" for i in range(16)], fontsize=8)
ax.set_xlabel("文本类名（锚）"); ax.set_ylabel("测试图（行按真实类）")
ax.set_title("相似度矩阵：行内最亮列=预测类；对角亮带越连续 zero-shot 越好", fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8, label="cos 相似度")
plt.tight_layout()
plt.savefig(FIGS / "fig3_simmat.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：错例 4 张（预测错且最高分是近邻类的典型情况）
wrong = [i for i in range(len(preds)) if preds[i] != int(yte[i])][:4]
fig, axes = plt.subplots(1, 4, figsize=(9.6, 2.7))
for ax, i in zip(axes, wrong):
    ax.imshow(unnormalize(Xte[i]))
    top2 = np.argsort(sims[i])[::-1][:2]
    ax.set_title(f"true {CIFAR10_CLASSES[int(yte[i])]}\npred {CIFAR10_CLASSES[preds[i]]} ({sims[i].max():.2f})\n2nd {CIFAR10_CLASSES[top2[1]]} ({sims[i][top2[1]]:.2f})", fontsize=8, color="#C0392B")
    ax.axis("off")
plt.suptitle("CLIP zero-shot 错例：最高/次高 cos 咬得极近，类名表征不够分化", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig4_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"zero-shot {acc_zs:.4f} vs 监督 {acc_sup:.4f}")


## 6. 总结与下一步

**本项目收获**

1. 双塔（图像 ViT + 文本 char-Encoder）+ InfoNCE 从零闭环，投影后单位向量 + 可学习温度
2. zero-shot 范式实测：不训分类头，`cos(图, 类名)` argmax 即分类，acc 见 fig2
3. 与监督 ViT 同台（同数据同预算）：小数据 CLIP 输给监督基线（真实结论：CLIP 靠海量图文对吃饭），但赢在**免标注扩展新类**（10 类→任意新 prompt 零样本可用）
4. 相似度矩阵/错例可视化：失败模式=近邻类 cos 咬太近（bird vs plane）

**下一步**：`03_Multimodal_Experience`——把本章的图像塔接上语言模型（视觉编码器+投影层+LLM 三段式），BLIP/LLaVA 只做推理体验不训练。